In [1]:
from train3 import make_env_fn
from evaluate import EvalInsulinPolicy, evaluate_insulin_policy
import argparse
import json
from dataclasses import asdict, dataclass
from pathlib import Path

import matplotlib.pyplot as plt
from stable_baselines3 import PPO
from stable_baselines3.common.callbacks import BaseCallback, CheckpointCallback, EvalCallback
from stable_baselines3.common.vec_env import DummyVecEnv, VecMonitor


patient="adult#001"
meals = "7:45,12:70,16:15,18:80,23:10"

meals = [
    (7 * 60, 45.0),
    (12 * 60, 70.0),
    (16 * 60, 15.0),
    (18 * 60, 80.0),
    (23 * 60, 10.0),
]

eval_env = DummyVecEnv([
    make_env_fn(
        env_id="simglucose-spid-eval-v0",
        patient=patient,
        meals=meals,
        max_episode_steps=480,
        seed=None,
        scenario_mode="fixed",
        time_std_multiplier=1,
        include_snacks=True,
        reward_type="smooth",
        warning_window_min=20,
        insulin_tau_min=55,
        sample_time_min=3,
        max_insulin_action=5
    )
])

eval_env = VecMonitor(eval_env)
model = PPO.load(r"C:\GitHub\GGSpeciale\simglucose_singlepatient\teacher_models\smooth\adult-001\models\best\best_model.zip", env=eval_env)

c:\Users\sofie\anaconda3\envs\thesis-1\lib\site-packages\gym\envs\registration.py:2: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


TypeError: make_env_fn() missing 5 required positional arguments: 'amount_noise_std_fraction', 'actual_time_noise_std_min', 'actual_time_noise_clip_min', 'shield_bg_threshold', and 'use_bb_warmup'

In [ ]:
evaluate_insulin_policy(model, eval_env, save_path="./logs/test/")

In [ ]:

evalCallback = EvalInsulinPolicy(eval_env, 
                                 eval_freq=1, 
                                 n_eval_episodes=20, 
                                 deterministic=False)

model.learn(
    total_timesteps=1,
    callback=[
        evalCallback,
    ],
    progress_bar=True
)